In [1]:
# =========================
# WEATHER STRESS PIPELINE
# =========================

import requests
import pandas as pd
import numpy as np
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# -------------------------
# CONFIG
# -------------------------
LAT = 17.384
LON = 78.4564

FORECAST_URL = f"https://api.open-meteo.com/v1/forecast?latitude={LAT}&longitude={LON}&hourly=temperature_2m,relative_humidity_2m,precipitation,precipitation_probability,et0_fao_evapotranspiration,vapour_pressure_deficit,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,soil_moisture_3_to_9cm,wind_speed_10m,cloud_cover&timezone=auto"

HIST_URL = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT}&longitude={LON}&start_date=2019-01-08&end_date=2025-12-31&daily=temperature_2m_mean,precipitation_sum,et0_fao_evapotranspiration,wind_speed_10m_max,shortwave_radiation_sum&hourly=vapour_pressure_deficit,relative_humidity_2m,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_temperature_0_to_7cm&timezone=auto"

# -------------------------
# HELPERS
# -------------------------
def fetch_json(url):
    return requests.get(url).json()

def hourly_to_daily(df, agg_map):
    df["date"] = pd.to_datetime(df["time"]).dt.date
    return df.groupby("date").agg(agg_map).reset_index()

# -------------------------
# LOAD HISTORICAL DATA
# -------------------------
hist = fetch_json(HIST_URL)

daily_hist = pd.DataFrame(hist["daily"])
daily_hist["date"] = pd.to_datetime(daily_hist["time"]).dt.date
daily_hist = daily_hist.drop(columns=["time"])

hourly_hist = pd.DataFrame(hist["hourly"])

agg_hist = hourly_to_daily(
    hourly_hist,
    {
        "vapour_pressure_deficit": "mean",
        "relative_humidity_2m": "mean",
        "soil_moisture_0_to_7cm": "mean",
        "soil_moisture_7_to_28cm": "mean",
        "soil_temperature_0_to_7cm": "mean",
    },
)

train_df = daily_hist.merge(agg_hist, on="date", how="inner")

# -------------------------
# FEATURE ENGINEERING
# -------------------------
train_df["water_deficit"] = train_df["et0_fao_evapotranspiration"] - train_df["precipitation_sum"]
train_df["root_zone_moisture"] = train_df[["soil_moisture_0_to_7cm","soil_moisture_7_to_28cm"]].mean(axis=1)
train_df["heat_index"] = train_df["temperature_2m_mean"] * train_df["vapour_pressure_deficit"]
train_df["rain_3d"] = train_df["precipitation_sum"].rolling(3).sum()

# -------------------------
# LABEL CREATION (RULE BASED)
# -------------------------
# =========================
# STRESS PROXY LABELS
# =========================

# Rolling climatology (30-day baseline)
train_df["temp_anomaly"] = train_df["temperature_2m_mean"] - train_df["temperature_2m_mean"].rolling(30).mean()
train_df["rain_anomaly"] = train_df["precipitation_sum"] - train_df["precipitation_sum"].rolling(30).mean()
train_df["soil_moisture_anomaly"] = train_df["root_zone_moisture"] - train_df["root_zone_moisture"].rolling(30).mean()

# Proxy stress definition (more realistic)
train_df["stress"] = (
    (train_df["temp_anomaly"] > 2) &
    (train_df["rain_anomaly"] < -1) &
    (train_df["soil_moisture_anomaly"] < -0.05)
).astype(int)

train_df = train_df.dropna()

# -------------------------
# MODEL TRAINING
# -------------------------
features = [
    "temperature_2m_mean",
    "precipitation_sum",
    "relative_humidity_2m",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
    "root_zone_moisture",
    "water_deficit",
    "heat_index",
    "rain_3d"
]

X = train_df[features]
y = train_df["stress"]

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, preds))

# -------------------------
# FORECAST INFERENCE
# -------------------------
forecast = fetch_json(FORECAST_URL)
hourly_fc = pd.DataFrame(forecast["hourly"])

fc_daily = hourly_to_daily(
    hourly_fc,
    {
        "temperature_2m": "mean",
        "relative_humidity_2m": "mean",
        "precipitation": "sum",
        "precipitation_probability": "max",
        "et0_fao_evapotranspiration": "sum",
        "vapour_pressure_deficit": "mean",
        "soil_moisture_0_to_1cm": "mean",
        "soil_moisture_1_to_3cm": "mean",
        "soil_moisture_3_to_9cm": "mean",
        "wind_speed_10m": "max",
        "cloud_cover": "mean",
    },
)

fc_daily["root_zone_moisture"] = fc_daily[
    ["soil_moisture_0_to_1cm","soil_moisture_1_to_3cm","soil_moisture_3_to_9cm"]
].mean(axis=1)

fc_daily["water_deficit"] = fc_daily["et0_fao_evapotranspiration"] - fc_daily["precipitation"]
fc_daily["heat_index"] = fc_daily["temperature_2m"] * fc_daily["vapour_pressure_deficit"]
fc_daily["rain_3d"] = fc_daily["precipitation"].rolling(3).sum()

fc_daily = fc_daily.dropna()

X_fc = fc_daily.rename(columns={
    "temperature_2m":"temperature_2m_mean",
    "precipitation":"precipitation_sum"
})[features]

fc_daily["stress_probability"] = model.predict_proba(X_fc)[:,1]

# -------------------------
# ADVISORY RULES
# -------------------------
fc_daily["stress_level"] = pd.cut(
    fc_daily["stress_probability"],
    bins=[0,0.33,0.66,1],
    labels=["Low","Medium","High"]
)

fc_daily["spray_window"] = (
    (fc_daily["precipitation_probability"] < 40) &
    (fc_daily["wind_speed_10m"] < 15)
)

fc_daily["irrigation_needed"] = (
    (fc_daily["water_deficit"] > 2) &
    (fc_daily["root_zone_moisture"] < 0.25)
)

print(fc_daily[[
    "date",
    "stress_probability",
    "stress_level",
    "spray_window",
    "irrigation_needed"
]])

ROC-AUC: 0.4686878727634195
         date  stress_probability stress_level  spray_window  \
2  2026-03-02            0.000429          Low          True   
3  2026-03-03            0.000411          Low          True   
4  2026-03-04            0.000130          Low          True   
5  2026-03-05            0.000115          Low          True   
6  2026-03-06            0.000126          Low          True   

   irrigation_needed  
2               True  
3               True  
4               True  
5               True  
6               True  


In [2]:
# =========================
# WEATHER BUSINESS METRICS (CLEAN)
# =========================

stress_pred_binary = (preds >= 0.5).astype(int)
stress_accuracy = float(np.mean(stress_pred_binary == y_test.values) * 100)

actionable_rate = float(
    np.mean(
        (fc_daily["spray_window"] == True) |
        (fc_daily["irrigation_needed"] == True)
    ) * 100
)

risk_lead_time = int(len(fc_daily))

print("\n☁️ WEATHER BUSINESS METRICS")
print(f"Stress Detection Accuracy: {stress_accuracy:.2f}%")
print(f"Actionable Advisory Rate: {actionable_rate:.2f}%")
print(f"Risk Lead Time: {risk_lead_time} days")


☁️ WEATHER BUSINESS METRICS
Stress Detection Accuracy: 99.60%
Actionable Advisory Rate: 100.00%
Risk Lead Time: 5 days


In [ ]:
import joblib

model_path = "stress_model.pkl"
joblib.dump(model, model_path)

['stress_model.pkl']

In [ ]:
import json
from google.colab import files

model_config = {
    "model_name": "weather_stress_xgb",
    "version": "1.0.0",
    "created_at": datetime.utcnow().isoformat(),
    "features": features,
    "target": "stress_probability",
    "stress_levels": {
        "low": [0, 0.33],
        "medium": [0.33, 0.66],
        "high": [0.66, 1]
    },
    "advisory_rules": {
        "spray_window": {
            "precipitation_probability_max": 40,
            "wind_speed_max": 15
        },
        "irrigation_needed": {
            "water_deficit_threshold": 2,
            "soil_moisture_threshold": 0.25
        }
    }
}

config_path = "model_config.json"

with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)

/tmp/ipython-input-1406270273.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [ ]:
files.download(model_path)
files.download(config_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>